## Crop Yield Prediction: Baseline and Core Models

This notebook uses the merged features in `data/processed/crop_yield_ml_features.csv` to train and compare multiple models:

- Linear Regression (baseline)
- Decision Tree
- Random Forest
- XGBoost (if `xgboost` is installed)
- SVR
- LightGBM (if `lightgbm` is installed)
- Stacking Ensemble

Metrics:
- R² Score
- RMSE
- MAE
- 5-fold cross-validation estimates for all metrics


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GroupKFold, TimeSeriesSplit, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.svm import SVR

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("xgboost not installed; XGBoost model will be skipped.")

try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False
    print("lightgbm not installed; LightGBM model will be skipped.")

pd.set_option("display.max_columns", None)


xgboost not installed; XGBoost model will be skipped.
lightgbm not installed; LightGBM model will be skipped.


In [3]:
# 1. Load merged feature dataset
data_path = "../data/processed/crop_yield_ml_features.csv"
df = pd.read_csv(data_path)
print("Shape:", df.shape)
df.head()

Shape: (19689, 16)


,crop,year,season,state,area,production,fertilizer,pesticide,yield,N,P,K,pH,avg_temp_c,total_rainfall_mm,avg_humidity_percent
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,7024878.38,22882.34,0.796087,60,18,38,5.8,22.41,1468.92,70.71
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,631643.29,2057.47,0.710435,60,18,38,5.8,22.41,1468.92,70.71
2,Castor seed,1997,Kharif,Assam,796.0,22,75755.32,246.76,0.238333,60,18,38,5.8,22.41,1468.92,70.71
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,1870661.52,6093.36,5238.051739,60,18,38,5.8,22.41,1468.92,70.71
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,165500.63,539.09,0.420909,60,18,38,5.8,22.41,1468.92,70.71


In [4]:
# 2. Define features and target
# IMPORTANT: `production` is often used to compute yield (yield ≈ production/area).
# Including it can cause target leakage and unrealistically high scores.
INCLUDE_LEAKY_PRODUCTION = False
INCLUDE_AREA = True
USE_LOG_TARGET = False  # try True if yield is extremely heavy-tailed

target_col = "yield"
y_raw = df[target_col].astype(float).values

feature_cols = [
    "crop", "season", "state",  # categorical
    "year",
    *( ["area"] if INCLUDE_AREA else [] ),
    *( ["production"] if INCLUDE_LEAKY_PRODUCTION else [] ),
    "fertilizer", "pesticide",
    "N", "P", "K", "pH",
    "avg_temp_c", "total_rainfall_mm", "avg_humidity_percent",
]
X = df[feature_cols].copy()

categorical_cols = ["crop", "season", "state"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

# Leakage diagnostic (always computed from raw df)
ratio = (df["production"].astype(float) / df["area"].astype(float)).replace([np.inf, -np.inf], np.nan)
valid = ratio.notna() & df[target_col].notna()
if valid.any():
    corr = np.corrcoef(df.loc[valid, target_col].astype(float), ratio.loc[valid].astype(float))[0, 1]
    median_abs_err = np.median(np.abs(df.loc[valid, target_col].astype(float) - ratio.loc[valid].astype(float)))
    print(f"Leakage check: corr(yield, production/area) = {corr:.6f}")
    print(f"Leakage check: median |yield - production/area| = {median_abs_err:.6f}")

print("Categorical features:", categorical_cols)
print("Numeric features:", numeric_cols)

# Target used for training
if USE_LOG_TARGET:
    y = np.log1p(y_raw)
else:
    y = y_raw


Categorical features: ['crop', 'season', 'state']
Numeric features: ['year', 'area', 'production', 'fertilizer', 'pesticide', 'N', 'P', 'K', 'pH', 'avg_temp_c', 'total_rainfall_mm', 'avg_humidity_percent']


In [5]:
# 3. Split strategy
# For forecasting, prefer a time-based split (train on early years, test on later years).
SPLIT_MODE = "time"  # "time" or "random"
TEST_START_YEAR = 2017

if SPLIT_MODE == "time":
    train_mask = X["year"] < TEST_START_YEAR
    test_mask = X["year"] >= TEST_START_YEAR

    X_train, X_test = X.loc[train_mask].copy(), X.loc[test_mask].copy()
    y_train, y_test = y[train_mask], y[test_mask]

    # CV that prevents mixing years across folds
    cv = GroupKFold(n_splits=5)
    cv_groups = X_train["year"].values
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_groups = None

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train years:", int(X_train["year"].min()), "to", int(X_train["year"].max()))
print("Test years:", int(X_test["year"].min()), "to", int(X_test["year"].max()))

((15751, 15), (3938, 15))

In [6]:
# 4. Preprocessing: one-hot encode categoricals, scale numerics
categorical_transformer = OneHotEncoder(handle_unknown="ignore")
numeric_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_transformer, categorical_cols),
        ("numeric", numeric_transformer, numeric_cols),
    ]
)
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numeric', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_na

In [7]:
# Helper function to evaluate models
# Uses the global split (X_train/X_test) + CV strategy (cv, cv_groups) defined above.
def evaluate_model(name, model):
    print(f"\n=== {name} ===")
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])

    scoring = {
        "r2": "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
    }

    cv_results = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        groups=cv_groups,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    # Fit on full training set and evaluate on hold-out test set
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    # If we trained on log1p(yield), convert predictions back for metrics
    if USE_LOG_TARGET:
        y_test_eval = y_raw[X_test.index]
        y_pred_eval = np.expm1(y_pred)
    else:
        y_test_eval = y_raw[X_test.index]
        y_pred_eval = y_pred

    r2 = r2_score(y_test_eval, y_pred_eval)
    rmse = np.sqrt(mean_squared_error(y_test_eval, y_pred_eval))
    mae = mean_absolute_error(y_test_eval, y_pred_eval)

    summary = {
        "model": name,
        "split_mode": SPLIT_MODE,
        "use_log_target": USE_LOG_TARGET,
        "include_production": INCLUDE_LEAKY_PRODUCTION,
        "include_area": INCLUDE_AREA,
        "test_r2": r2,
        "test_rmse": rmse,
        "test_mae": mae,
        "cv_r2_mean": cv_results["test_r2"].mean(),
        "cv_r2_std": cv_results["test_r2"].std(),
        "cv_rmse_mean": -cv_results["test_rmse"].mean(),  # invert sign
        "cv_rmse_std": cv_results["test_rmse"].std(),
        "cv_mae_mean": -cv_results["test_mae"].mean(),
        "cv_mae_std": cv_results["test_mae"].std(),
    }

    print("Test R2:", r2)
    print("Test RMSE:", rmse)
    print("Test MAE:", mae)
    print("CV R2 (mean ± std):", summary["cv_r2_mean"], "+/-", summary["cv_r2_std"])
    print("CV RMSE (mean):", summary["cv_rmse_mean"])
    print("CV MAE (mean):", summary["cv_mae_mean"])

    return summary


In [8]:
# 5. Define models
models = []

# 1. Baseline: Linear Regression
models.append(("Linear Regression", LinearRegression()))

# 2. Core models
models.append(("Decision Tree", DecisionTreeRegressor(random_state=42)))
models.append(("Random Forest", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)))

if XGB_AVAILABLE:
    models.append((
        "XGBoost",
        XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            n_jobs=-1,
            random_state=42,
        ),
    ))

models.append(("SVR (RBF)", SVR(kernel="rbf", C=10.0, epsilon=0.1)))

# 3. Advanced: LightGBM (if available)
if LGBM_AVAILABLE:
    models.append((
        "LightGBM",
        LGBMRegressor(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
        ),
    ))

models

[('Linear Regression', LinearRegression()),
 ('Decision Tree', DecisionTreeRegressor(random_state=42)),
 ('Random Forest',
  RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)),
 ('SVR (RBF)', SVR(C=10.0))]

In [9]:
# 6. Run all models and collect results
results = []
for name, mdl in models:
    summary = evaluate_model(name, mdl)
    results.append(summary)

results_df = pd.DataFrame(results)
results_df_sorted = results_df.sort_values(by="test_r2", ascending=False)
results_df_sorted


=== Linear Regression ===
Test R2: 0.8021905687726195
Test RMSE: 398.1115446130519
Test MAE: 63.12042603826624
CV R2 (mean ± std): 0.8444852057838368 +/- 0.01982799576891547
CV RMSE (mean): 340.3106747367403
CV MAE (mean): 57.56467450111684

=== Decision Tree ===
Test R2: 0.9755142850453885
Test RMSE: 140.0675941908516
Test MAE: 8.242819285758507
CV R2 (mean ± std): 0.9577184153973637 +/- 0.024322218835269128
CV RMSE (mean): 173.67220272278075
CV MAE (mean): 11.344025793892397

=== Random Forest ===
Test R2: 0.9842901942731325
Test RMSE: 112.19323784582151
Test MAE: 7.830418649389032
CV R2 (mean ± std): 0.975996804247956 +/- 0.011600226961899612
CV RMSE (mean): 132.19223012365978
CV MAE (mean): 8.593372232687262

=== SVR (RBF) ===
Test R2: 0.015501458834129345
Test RMSE: 888.155098012752
Test MAE: 75.23835360605639
CV R2 (mean ± std): 0.011775119203920714 +/- 0.0030899856978886956
CV RMSE (mean): 862.8878922545349
CV MAE (mean): 76.31137223992779


,model,test_r2,test_rmse,test_mae,cv_r2_mean,cv_r2_std,cv_rmse_mean,cv_rmse_std,cv_mae_mean,cv_mae_std
2,Random Forest,0.984290,112.193238,7.830419,0.975997,0.011600,132.192230,39.238528,8.593372,1.586949
1,Decision Tree,0.975514,140.067594,8.242819,0.957718,0.024322,173.672203,67.901963,11.344026,3.478572
0,Linear Regression,0.802191,398.111545,63.120426,0.844485,0.019828,340.310675,33.201316,57.564675,3.346669
3,SVR (RBF),0.015501,888.155098,75.238354,0.011775,0.003090,862.887892,103.186224,76.311372,14.469440


In [10]:
# 7. Stacking Ensemble (advanced)
base_estimators = []
for name, mdl in models:
    if name in ["Linear Regression"]:
        continue
    base_estimators.append((name.replace(" ", "_"), mdl))

final_estimator = LinearRegression()

stack_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=final_estimator,
    n_jobs=-1,
)

stack_summary = evaluate_model("Stacking Ensemble", stack_model)
results.append(stack_summary)
results_full = pd.DataFrame(results).sort_values(by="test_r2", ascending=False)
results_full


=== Stacking Ensemble ===


KeyboardInterrupt: 

In [ ]:
# 8. Visual comparison of model performance
plt.figure(figsize=(10, 5))
sns.barplot(
    data=results_full,
    x="model",
    y="test_r2",
    order=results_full.sort_values("test_r2", ascending=False)["model"],
)
plt.xticks(rotation=45, ha="right")
plt.title("Model comparison (Test R²)")
plt.tight_layout()
plt.show()

results_full